# Problem Set 8: Legal Drinking Age and Mortality

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due 11/18 (beginning of Wednesday lecture)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Regression discontinuity at the minimum legal drinking age.

**Before you do anything else:** go to **File → Save a copy in Drive**.

> 📝 **DRAFT for instructor review** — questions and emphasis to be finalized with Prof. Schwandt.

## Part 0: Set-up

Can we measure the effect of alcohol access on mortality? People who drink differ from people who don't in a hundred ways — but the **minimum legal drinking age** creates a natural experiment: at exactly age 21, alcohol becomes legal. If nothing else changes discontinuously at 21, then any **jump** in death rates right at the 21st birthday can be attributed to legal alcohol access. This is a **regression discontinuity (RD)** design, following Carpenter and Dobkin (2009).

The data contains death rates (deaths per 100,000 person-years) for Americans in 50 two-month age cells from 19 to 23, by cause of death: `all`, `mva` (motor vehicle accidents), `alcohol`, `internal` (internal causes like cancer or heart disease), `suicide`, `homicide`, and others. `agecell` is the age midpoint of each cell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

In [ ]:
mlda = pd.read_stata("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/AEJfigs.dta")
mlda = mlda.dropna(subset=["all"]).copy()

# center age at 21 and create the "over 21" indicator
mlda["age"] = mlda.agecell - 21
mlda["over21"] = (mlda.agecell >= 21).astype(int)

print(mlda.shape)
mlda[["agecell", "age", "over21", "all", "mva", "internal", "alcohol"]].head()

## Part 1: Look at the raw data

Before any regression: plot the all-cause death rate against age.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(mlda.agecell, mlda["all"])
plt.axvline(21, color="gray", linestyle="--")
plt.xlabel("Age"); plt.ylabel("Deaths per 100,000 person-years")
plt.title("All-cause mortality around age 21")
plt.show()

**Question 1.** Describe what you see. Is there anything special happening at age 21, compared to the general trend?

**Answer:** *[your answer here]*

## Part 2: The RD estimate

We regress the death rate on the `over21` indicator plus a linear trend in (centered) age, allowing the trend to differ on each side of the cutoff. The coefficient on `over21` is the **jump at 21** — our causal estimate.

In [ ]:
rd_all = smf.ols("Q('all') ~ over21 + age + age:over21", mlda).fit()
print(rd_all.summary().tables[1])

# plot the data with the fitted lines on both sides of 21
plt.figure(figsize=(8, 5))
plt.scatter(mlda.agecell, mlda["all"])
for side in [0, 1]:
    sub = mlda[mlda.over21 == side]
    plt.plot(sub.agecell, rd_all.predict(sub), color="C1")
plt.axvline(21, color="gray", linestyle="--")
plt.xlabel("Age"); plt.ylabel("Deaths per 100,000 person-years")
plt.title("RD: all-cause mortality at the drinking age")
plt.show()

print(f"Estimated jump at 21: {rd_all.params['over21']:.2f} deaths per 100,000 person-years")

**Question 2a.** How large is the jump at 21, and is it statistically significant? Interpret it in words.

**Answer:** *[your answer here]*

**Question 2b.** The key **identifying assumption** of RD is that all *other* determinants of mortality evolve smoothly through age 21 — only alcohol access jumps. Name one thing that could violate this assumption here, and say whether you find it plausible.

**Answer:** *[your answer here]*

## Part 3: Where does the effect come from? (And a placebo)

If the jump is really about alcohol, it should show up in alcohol-related causes — motor vehicle accidents especially — and **not** in internal causes (cancer, heart disease), which have no business responding to a birthday. Internal causes are our **placebo test**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
rd_results = {}
for ax, cause, label in [(axes[0], "mva", "Motor vehicle accidents"),
                         (axes[1], "internal", "Internal causes (placebo)")]:
    r = smf.ols(f"{cause} ~ over21 + age + age:over21", mlda).fit()
    rd_results[cause] = r
    ax.scatter(mlda.agecell, mlda[cause])
    for side in [0, 1]:
        sub = mlda[mlda.over21 == side]
        ax.plot(sub.agecell, r.predict(sub), color="C1")
    ax.axvline(21, color="gray", linestyle="--")
    ax.set_xlabel("Age"); ax.set_title(label)
axes[0].set_ylabel("Deaths per 100,000 person-years")
plt.show()

for cause, r in rd_results.items():
    print(f"{cause:10s} jump at 21: {r.params['over21']:6.2f}  (t = {r.tvalues['over21']:.2f})")

**Question 3.** Compare the two panels. What do you find for motor vehicle accidents, and what for internal causes? Why does the internal-causes result *strengthen* your confidence in the causal interpretation of the MVA and all-cause results?

**Answer:** *[your answer here]*

## Part 4: Robustness

RD estimates can be sensitive to how we model the trend. Re-estimate the all-cause jump using a **quadratic** in age on each side (add `I(age**2)` and `I(age**2):over21` to the formula) in the cell below, and compare the jump to your linear estimate.

In [ ]:
# Your quadratic specification here
rd_quad = smf.ols("Q('all') ~ over21 + age + age:over21 + I(age**2) + I(age**2):over21", mlda).fit()
print(f"Linear jump   : {rd_all.params['over21']:.2f}")
print(f"Quadratic jump: {rd_quad.params['over21']:.2f}  (t = {rd_quad.tvalues['over21']:.2f})")

**Question 4.** Does the estimate change much with the quadratic? What does that tell you about the robustness of this particular RD?

**Answer:** *[your answer here]*

**Question 5.** Based on your estimates, briefly discuss: what would you predict would happen to mortality if the drinking age were lowered to 18? What does this design let you say — and, just as importantly, what does it *not* let you say (think: external validity away from age 21)?

**Answer:** *[your answer here]*

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS08"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")